In [0]:
# Carregamento das tabelas da camada Silver

df_acidentes_silver = spark.table("workspace.silver.acidentes")
df_localidade_silver = spark.table("workspace.silver.localidade")
df_tipo_veiculo_silver = spark.table("workspace.silver.tipo_veiculo")
df_vitimas_silver = spark.table("workspace.silver.vitimas")

print("Tabelas Silver carregadas com sucesso.")

Tabelas Silver carregadas com sucesso.


In [0]:
from pyspark.sql import functions as F

# Construção da dimensão Tempo a partir das datas dos acidentes
dim_tempo = (
    df_acidentes_silver
    .select("data_acidente")
    .filter(F.col("data_acidente").isNotNull())
    .distinct()
    .withColumn(
        "id_tempo",
        F.date_format(F.col("data_acidente"), "yyyyMMdd").cast("int")
    )
    .withColumn("ano", F.year(F.col("data_acidente")))
    .withColumn("mes", F.month(F.col("data_acidente")))
    .withColumn("dia", F.dayofmonth(F.col("data_acidente")))
    .withColumn("dia_semana_num", F.dayofweek(F.col("data_acidente")))
    .withColumn(
        "dia_semana",
        F.when(F.col("dia_semana_num") == 1, "DOMINGO")
         .when(F.col("dia_semana_num") == 2, "SEGUNDA-FEIRA")
         .when(F.col("dia_semana_num") == 3, "TERCA-FEIRA")
         .when(F.col("dia_semana_num") == 4, "QUARTA-FEIRA")
         .when(F.col("dia_semana_num") == 5, "QUINTA-FEIRA")
         .when(F.col("dia_semana_num") == 6, "SEXTA-FEIRA")
         .when(F.col("dia_semana_num") == 7, "SABADO")
    )
    .withColumn("trimestre", F.quarter(F.col("data_acidente")))
    .orderBy("data_acidente")
)

print("Dimensão Tempo construída com sucesso.")

Dimensão Tempo construída com sucesso.


In [0]:
# Inspeção da dimensão Tempo
display(
    dim_tempo
    .select(
        "id_tempo",
        "data_acidente",
        "ano",
        "mes",
        "dia",
        "dia_semana_num",
        "dia_semana",
        "trimestre"
    )
    .limit(20)
)

print("\nSchema - Dimensão Tempo:")
dim_tempo.printSchema()

id_tempo,data_acidente,ano,mes,dia,dia_semana_num,dia_semana,trimestre
20180101,2018-01-01,2018,1,1,2,SEGUNDA-FEIRA,1
20180102,2018-01-02,2018,1,2,3,TERCA-FEIRA,1
20180103,2018-01-03,2018,1,3,4,QUARTA-FEIRA,1
20180104,2018-01-04,2018,1,4,5,QUINTA-FEIRA,1
20180105,2018-01-05,2018,1,5,6,SEXTA-FEIRA,1
20180106,2018-01-06,2018,1,6,7,SABADO,1
20180107,2018-01-07,2018,1,7,1,DOMINGO,1
20180108,2018-01-08,2018,1,8,2,SEGUNDA-FEIRA,1
20180109,2018-01-09,2018,1,9,3,TERCA-FEIRA,1
20180110,2018-01-10,2018,1,10,4,QUARTA-FEIRA,1



Schema - Dimensão Tempo:
root
 |-- data_acidente: date (nullable = true)
 |-- id_tempo: integer (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- dia_semana_num: integer (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- trimestre: integer (nullable = true)



In [0]:
# Persistência da dimensão Tempo na camada Gold
(
    dim_tempo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_tempo")
)

print("Tabela workspace.gold.dim_tempo gravada com sucesso.")

Tabela workspace.gold.dim_tempo gravada com sucesso.


In [0]:
# Inspeção dos horários existentes na camada Silver
df_horarios = (
    df_acidentes_silver
    .select("hora_acidente")
    .filter(F.col("hora_acidente").isNotNull())
    .distinct()
    .orderBy("hora_acidente")
)

display(df_horarios.limit(20))

hora_acidente
2018-01-01T00:00:00.000Z
2018-01-01T00:10:00.000Z
2018-01-01T00:30:00.000Z
2018-01-01T00:40:00.000Z
2018-01-01T01:40:00.000Z
2018-01-01T02:00:00.000Z
2018-01-01T03:00:00.000Z
2018-01-01T03:32:00.000Z
2018-01-01T04:30:00.000Z
2018-01-01T08:30:00.000Z


In [0]:
# Construção da dimensão Horário a partir dos horários dos acidentes
dim_horario = (
    df_acidentes_silver
    .filter(F.col("hora_acidente").isNotNull())
    .select(
        F.hour("hora_acidente").alias("hora"),
        F.minute("hora_acidente").alias("minuto"),
        F.second("hora_acidente").alias("segundo")
    )
    .distinct()

    # Chave no formato HHMMSS
    .withColumn(
        "id_horario",
        (
            F.col("hora") * 10000 +
            F.col("minuto") * 100 +
            F.col("segundo")
        ).cast("int")
    )

    # Representação legível do horário
    .withColumn(
        "horario",
        F.format_string(
            "%02d:%02d:%02d",
            F.col("hora"),
            F.col("minuto"),
            F.col("segundo")
        )
    )

    # Faixa horária para análise
    .withColumn(
        "faixa_horaria",
        F.when((F.col("hora") >= 0) & (F.col("hora") < 6), "MADRUGADA")
         .when((F.col("hora") >= 6) & (F.col("hora") < 12), "MANHA")
         .when((F.col("hora") >= 12) & (F.col("hora") < 18), "TARDE")
         .otherwise("NOITE")
    )

    .select(
        "id_horario",
        "horario",
        "hora",
        "minuto",
        "segundo",
        "faixa_horaria"
    )
    .orderBy("id_horario")
)

print("Dimensão Horário construída com sucesso.")

Dimensão Horário construída com sucesso.


In [0]:
# Validação visual e do schema da dimensão Horário
display(
    dim_horario
    .select(
        "id_horario",
        "horario",
        "hora",
        "minuto",
        "segundo",
        "faixa_horaria"
    )
    .limit(30)
)

print("\nSchema - Dimensão Horário:")
dim_horario.printSchema()

id_horario,horario,hora,minuto,segundo,faixa_horaria
0,00:00:00,0,0,0,MADRUGADA
100,00:01:00,0,1,0,MADRUGADA
200,00:02:00,0,2,0,MADRUGADA
300,00:03:00,0,3,0,MADRUGADA
400,00:04:00,0,4,0,MADRUGADA
500,00:05:00,0,5,0,MADRUGADA
600,00:06:00,0,6,0,MADRUGADA
700,00:07:00,0,7,0,MADRUGADA
800,00:08:00,0,8,0,MADRUGADA
900,00:09:00,0,9,0,MADRUGADA



Schema - Dimensão Horário:
root
 |-- id_horario: integer (nullable = true)
 |-- horario: string (nullable = false)
 |-- hora: integer (nullable = true)
 |-- minuto: integer (nullable = true)
 |-- segundo: integer (nullable = true)
 |-- faixa_horaria: string (nullable = false)



In [0]:
# Persistência da dimensão Horário na camada Gold
(
    dim_horario.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_horario")
)

print("Tabela workspace.gold.dim_horario gravada com sucesso.")

Tabela workspace.gold.dim_horario gravada com sucesso.


In [0]:
# Inspeção dos bairros existentes na camada Silver

df_bairros_gold_base = (
    df_acidentes_silver
    .select("bairro_acidente")
    .groupBy("bairro_acidente")
    .count()
    .orderBy(F.desc("count"))
)

display(df_bairros_gold_base.limit(40))

bairro_acidente,count
null,26894
CAMPO GRANDE,1618
BARRA DA TIJUCA,1360
BANGU,1011
SANTA CRUZ,809
CENTRO,797
BONSUCESSO,776
RECREIO DOS BANDEIRANTES,772
GUARATIBA,750
REALENGO,725


In [0]:
# Verificação dos marcadores conhecidos de ausência/informação incompleta

df_bairros_ausencia = (
    df_acidentes_silver
    .filter(
        F.col("bairro_acidente").isNull()
        | F.col("bairro_acidente").isin(
            "NI",
            "SEM INFORMACAO",
            "NAO INFORMADO",
            "DESCONHECIDO"
        )
    )
    .groupBy("bairro_acidente")
    .count()
    .orderBy(F.desc("count"))
)

display(df_bairros_ausencia)

bairro_acidente,count
null,26894


In [0]:
# Construção da dimensão Bairro

# Bairros válidos existentes nos acidentes
dim_bairro_validos = (
    df_acidentes_silver
    .select(
        F.col("bairro_acidente").alias("bairro")
    )
    .filter(F.col("bairro").isNotNull())
    .distinct()
)

# Membro especial para preservar acidentes sem bairro informado
dim_bairro_nao_informado = spark.createDataFrame(
    [(-1, "NAO INFORMADO")],
    ["id_bairro", "bairro"]
)

# Chave determinística para os bairros válidos
dim_bairro_validos = (
    dim_bairro_validos
    .withColumn(
        "id_bairro",
        F.abs(F.xxhash64(F.col("bairro")))
    )
    .select(
        "id_bairro",
        "bairro"
    )
)

# Dimensão final
dim_bairro = (
    dim_bairro_nao_informado
    .unionByName(dim_bairro_validos)
)

print("Dimensão Bairro construída com sucesso.")

Dimensão Bairro construída com sucesso.


In [0]:
# Validação da dimensão Bairro

print("Membro especial para bairro não informado:")
display(
    dim_bairro
    .filter(F.col("id_bairro") == -1)
)

print("Amostra dos bairros:")
display(
    dim_bairro
    .orderBy("bairro")
    .limit(20)
)

# Quantidade total de registros
total_bairros = dim_bairro.count()

# Verificação de duplicidade da chave
chaves_duplicadas = (
    dim_bairro
    .groupBy("id_bairro")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Verificação de duplicidade do nome do bairro
bairros_duplicados = (
    dim_bairro
    .groupBy("bairro")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Quantidade de registros na dimensão: {total_bairros}")
print(f"Chaves id_bairro duplicadas: {chaves_duplicadas}")
print(f"Bairros duplicados: {bairros_duplicados}")

Membro especial para bairro não informado:


id_bairro,bairro
-1,NAO INFORMADO


Amostra dos bairros:


id_bairro,bairro
1207384447670883460,00000000
3495362634406624192,7 RIACHOS
4532742852458505499,ABLICAO
1395425764441353758,ABOLICAO
2739530482740225230,ACARI
2161512045191906124,AGUA SANTA
7873277181823551377,ALTO DA BOA VISTA
6451962876486310616,ANA GONZAGA
61959487178479528,ANCHETA
2078442611386165660,ANCHIETA


Quantidade de registros na dimensão: 311
Chaves id_bairro duplicadas: 0
Bairros duplicados: 0


In [0]:
# Persistência da dimensão Bairro na camada Gold
(
    dim_bairro.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_bairro")
)

print("Tabela workspace.gold.dim_bairro gravada com sucesso.")

Tabela workspace.gold.dim_bairro gravada com sucesso.


In [0]:
# Inspeção dos atributos de localização disponíveis na Silver

display(
    df_localidade_silver
    .select(
        "codigo_ibge",
        "municipio",
        "uf"
    )
    .distinct()
    .limit(20)
)

print("Colunas disponíveis em Localidade:")
for coluna in df_localidade_silver.columns:
    print(f"- {coluna}")

codigo_ibge,municipio,uf
3304557,RIO DE JANEIRO,RJ


Colunas disponíveis em Localidade:
- chv_localidade
- ano_referencia
- mes_referencia
- mes_ano_referencia
- regiao
- uf
- codigo_ibge
- municipio
- regiao_metropolitana
- qtde_habitantes
- frota_total
- frota_circulante
- _data_ingestao
- _arquivo_origem
- _sistema_origem


In [0]:
# Bairros da dimensão que poderão ser associados a Regiões Administrativas

df_bairros_para_regiao = (
    dim_bairro
    .filter(F.col("id_bairro") != -1)
    .select("id_bairro", "bairro")
)

print(
    "Quantidade de bairros para avaliação regional:",
    df_bairros_para_regiao.count()
)

display(
    df_bairros_para_regiao
    .orderBy("bairro")
    .limit(30)
)

Quantidade de bairros para avaliação regional: 310


id_bairro,bairro
1207384447670883460,00000000
3495362634406624192,7 RIACHOS
4532742852458505499,ABLICAO
1395425764441353758,ABOLICAO
2739530482740225230,ACARI
2161512045191906124,AGUA SANTA
7873277181823551377,ALTO DA BOA VISTA
6451962876486310616,ANA GONZAGA
61959487178479528,ANCHETA
2078442611386165660,ANCHIETA


In [0]:
# Extração da referência oficial de bairros e Regiões Administrativas
# Fonte: Instituto Pereira Passos / Prefeitura da Cidade do Rio de Janeiro

import requests

url_bairros_ipp = (
    "https://pgeo3.rio.rj.gov.br/arcgis/rest/services/"
    "Cartografia/Limites_administrativos/FeatureServer/4/query"
)

params = {
    "where": "1=1",
    "outFields": "nome,codbairro,regiao_adm,codra,area_plane",
    "returnGeometry": "false",
    "f": "json"
}

response = requests.get(
    url_bairros_ipp,
    params=params,
    timeout=60
)

response.raise_for_status()

dados_ipp = response.json()

if "error" in dados_ipp:
    raise Exception(dados_ipp["error"])

registros_ipp = [
    feature["attributes"]
    for feature in dados_ipp["features"]
]

df_bairros_ipp = spark.createDataFrame(registros_ipp)

print(f"Registros recebidos da fonte oficial: {len(registros_ipp)}")

display(
    df_bairros_ipp
    .select(
        "codbairro",
        "nome",
        "codra",
        "regiao_adm",
        "area_plane"
    )
    .orderBy("nome")
    .limit(30)
)

Registros recebidos da fonte oficial: 167


codbairro,nome,codra,regiao_adm,area_plane
070,Abolição,13,MEIER,3
111,Acari,25,PAVUNA,3
034,Alto da Boa Vista,8,TIJUCA,2
107,Anchieta,22,ANCHIETA,3
037,Andaraí,9,VILA ISABEL,2
116,Anil,16,JACAREPAGUA,4
166,Argentino,11,PENHA,3
097,Bancários,20,ILHA DO GOVERNADOR,3
141,Bangu,17,BANGU,5
165,Barra Olímpica,16,JACAREPAGUA,4


In [0]:
# Normalização utilizada SOMENTE para comparação entre as fontes

from pyspark.sql import functions as F

# Normaliza os bairros do RENAEST
bairros_renaest_match = (
    df_bairros_para_regiao
    .withColumn(
        "bairro_match",
        F.upper(
            F.translate(
                F.trim(F.col("bairro")),
                "ÁÀÃÂÉÊÍÓÔÕÚÜÇáàãâéêíóôõúüç",
                "AAAAEEIOOOUUCaaaaeeiooouuc"
            )
        )
    )
)

# Normaliza os bairros oficiais do IPP
bairros_ipp_match = (
    df_bairros_ipp
    .withColumn(
        "bairro_match",
        F.upper(
            F.translate(
                F.trim(F.col("nome")),
                "ÁÀÃÂÉÊÍÓÔÕÚÜÇáàãâéêíóôõúüç",
                "AAAAEEIOOOUUCaaaaeeiooouuc"
            )
        )
    )
)

# Cruzamento exato após normalização
comparacao_bairros = (
    bairros_renaest_match.alias("r")
    .join(
        bairros_ipp_match.alias("i"),
        F.col("r.bairro_match") == F.col("i.bairro_match"),
        "left"
    )
    .select(
        F.col("r.id_bairro"),
        F.col("r.bairro").alias("bairro_renaest"),
        F.col("i.codbairro"),
        F.col("i.nome").alias("bairro_ipp"),
        F.col("i.codra"),
        F.col("i.regiao_adm"),
        F.col("i.area_plane")
    )
)

total = comparacao_bairros.count()

encontrados = (
    comparacao_bairros
    .filter(F.col("bairro_ipp").isNotNull())
    .count()
)

nao_encontrados = (
    comparacao_bairros
    .filter(F.col("bairro_ipp").isNull())
    .count()
)

print(f"Bairros distintos do RENAEST avaliados: {total}")
print(f"Correspondências exatas encontradas: {encontrados}")
print(f"Sem correspondência: {nao_encontrados}")
print(f"Taxa de correspondência: {(encontrados / total) * 100:.2f}%")

print("\nValores sem correspondência:")
display(
    comparacao_bairros
    .filter(F.col("bairro_ipp").isNull())
    .select("bairro_renaest")
    .orderBy("bairro_renaest")
)

Bairros distintos do RENAEST avaliados: 310
Correspondências exatas encontradas: 157
Sem correspondência: 153
Taxa de correspondência: 50.65%

Valores sem correspondência:


bairro_renaest
00000000
7 RIACHOS
ABLICAO
ANA GONZAGA
ANCHETA
ATAFONA
BAIRRO ACARI
BAIRRO DISTRITO INDUSTRIAL
BAIRRO NAO CADASTRADO
BARA DA TIJUCA


In [0]:
# Avaliação da cobertura do mapeamento Bairro -> RA
# considerando a quantidade real de acidentes

acidentes_match = (
    df_acidentes_silver
    .withColumn(
        "bairro_match",
        F.upper(
            F.translate(
                F.trim(F.col("bairro_acidente")),
                "ÁÀÃÂÉÊÍÓÔÕÚÜÇáàãâéêíóôõúüç",
                "AAAAEEIOOOUUCaaaaeeiooouuc"
            )
        )
    )
)

acidentes_com_regiao = (
    acidentes_match.alias("a")
    .join(
        bairros_ipp_match.alias("i"),
        F.col("a.bairro_match") == F.col("i.bairro_match"),
        "left"
    )
)

total_acidentes = acidentes_com_regiao.count()

com_regiao = (
    acidentes_com_regiao
    .filter(F.col("i.codra").isNotNull())
    .count()
)

sem_regiao = total_acidentes - com_regiao

print(f"Total de acidentes: {total_acidentes}")
print(f"Acidentes associados a uma RA: {com_regiao}")
print(f"Acidentes sem associação a RA: {sem_regiao}")
print(f"Cobertura regional: {(com_regiao / total_acidentes) * 100:.2f}%")

print("\nPrincipais valores de bairro sem correspondência:")

display(
    acidentes_com_regiao
    .filter(F.col("i.codra").isNull())
    .groupBy("a.bairro_acidente")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

Total de acidentes: 55046
Acidentes associados a uma RA: 27075
Acidentes sem associação a RA: 27971
Cobertura regional: 49.19%

Principais valores de bairro sem correspondência:


bairro_acidente,count
null,26894
SAO CRISTOVAO,396
00000000,86
FREGUESIA,86
OSWALDO CRUZ,68
FREGUESIA JACAREPAGUA,58
BAIRRO NAO CADASTRADO,49
ILHA DO GOVERNADOR,48
MARIOPOLIS,23
SULACAP,17


In [0]:
# Diagnóstico temporal da ausência de associação com RA

diagnostico_ra_ano = (
    acidentes_com_regiao
    .withColumn(
        "status_ra",
        F.when(
            F.col("i.codra").isNotNull(),
            F.lit("COM_RA")
        )
        .when(
            F.col("a.bairro_acidente").isNull(),
            F.lit("SEM_BAIRRO")
        )
        .otherwise(
            F.lit("BAIRRO_NAO_ASSOCIADO")
        )
    )
    .groupBy(
        "a.ano_acidente",
        "status_ra"
    )
    .count()
)

display(
    diagnostico_ra_ano
    .orderBy(
        "ano_acidente",
        "status_ra"
    )
)

ano_acidente,status_ra,count
2018,SEM_BAIRRO,9683
2019,COM_RA,1
2019,SEM_BAIRRO,9448
2020,BAIRRO_NAO_ASSOCIADO,1
2020,COM_RA,14
2020,SEM_BAIRRO,6021
2021,BAIRRO_NAO_ASSOCIADO,242
2021,COM_RA,6269
2021,SEM_BAIRRO,407
2022,BAIRRO_NAO_ASSOCIADO,284


In [0]:
# Ranking dos valores de bairro que possuem informação,
# mas não encontraram correspondência exata no cadastro oficial do IPP

bairros_nao_associados = (
    acidentes_com_regiao
    .filter(
        F.col("i.codra").isNull()
        & F.col("a.bairro_acidente").isNotNull()
    )
    .groupBy("a.bairro_acidente")
    .count()
    .orderBy(F.desc("count"), F.asc("bairro_acidente"))
)

total_registros_nao_associados = (
    acidentes_com_regiao
    .filter(
        F.col("i.codra").isNull()
        & F.col("a.bairro_acidente").isNotNull()
    )
    .count()
)

total_valores_distintos = bairros_nao_associados.count()

print(
    f"Registros com bairro informado e sem associação: "
    f"{total_registros_nao_associados}"
)

print(
    f"Valores distintos envolvidos: "
    f"{total_valores_distintos}"
)

display(bairros_nao_associados)

Registros com bairro informado e sem associação: 1077
Valores distintos envolvidos: 153


bairro_acidente,count
SAO CRISTOVAO,396
00000000,86
FREGUESIA,86
OSWALDO CRUZ,68
FREGUESIA JACAREPAGUA,58
BAIRRO NAO CADASTRADO,49
ILHA DO GOVERNADOR,48
MARIOPOLIS,23
SULACAP,17
RECREIO,15


In [0]:
# Diagnóstico de similaridade textual
# NÃO realiza correção automática

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Valores problemáticos do RENAEST
nao_associados_match = (
    bairros_nao_associados
    .withColumn(
        "bairro_match",
        F.upper(
            F.translate(
                F.trim(F.col("bairro_acidente")),
                "ÁÀÃÂÉÊÍÓÔÕÚÜÇáàãâéêíóôõúüç",
                "AAAAEEIOOOUUCaaaaeeiooouuc"
            )
        )
    )
)

# Nomes oficiais IPP já normalizados em bairros_ipp_match

# Produto cartesiano pequeno:
# 153 valores RENAEST x 167 bairros oficiais
candidatos = (
    nao_associados_match.alias("r")
    .crossJoin(bairros_ipp_match.alias("i"))
    .withColumn(
        "distancia",
        F.levenshtein(
            F.col("r.bairro_match"),
            F.col("i.bairro_match")
        )
    )
)

# Melhor candidato oficial para cada valor do RENAEST
janela = (
    Window
    .partitionBy("r.bairro_acidente")
    .orderBy(
        F.asc("distancia"),
        F.asc("i.nome")
    )
)

melhores_candidatos = (
    candidatos
    .withColumn("ordem", F.row_number().over(janela))
    .filter(F.col("ordem") == 1)
    .select(
        F.col("r.bairro_acidente").alias("bairro_renaest"),
        F.col("r.count").alias("qtd_acidentes"),
        F.col("i.nome").alias("bairro_oficial_candidato"),
        F.col("i.codbairro"),
        F.col("i.codra"),
        F.col("i.regiao_adm"),
        F.col("distancia")
    )
    .orderBy(
        F.desc("qtd_acidentes"),
        F.asc("distancia")
    )
)

display(melhores_candidatos)

bairro_renaest,qtd_acidentes,bairro_oficial_candidato,codbairro,codra,regiao_adm,distancia
SAO CRISTOVAO,396,Santo Cristo,003,1,PORTUARIA,5
FREGUESIA,86,Mangueira,011,7,SAO CRISTOVAO,5
00000000,86,Abolição,070,13,MEIER,8
OSWALDO CRUZ,68,Osvaldo Cruz,088,15,MADUREIRA,1
FREGUESIA JACAREPAGUA,58,Freguesia (Jacarepaguá),120,16,JACAREPAGUA,2
BAIRRO NAO CADASTRADO,49,Barra da Tijuca,128,24,BARRA DA TIJUCA,13
ILHA DO GOVERNADOR,48,Ilha de Guaratiba,164,26,GUARATIBA,9
MARIOPOLIS,23,Higienópolis,050,12,INHAUMA,5
SULACAP,17,Lapa,161,2,CENTRO,4
RECREIO,15,Realengo,139,33,REALENGO,4


In [0]:
# Avaliação das correspondências muito próximas
# Apenas diagnóstico - nenhuma correção é aplicada

candidatos_distancia_1 = (
    melhores_candidatos
    .filter(F.col("distancia") == 1)
    .orderBy(F.desc("qtd_acidentes"))
)

qtd_valores_distancia_1 = candidatos_distancia_1.count()

qtd_acidentes_distancia_1 = (
    candidatos_distancia_1
    .agg(F.sum("qtd_acidentes").alias("total"))
    .first()["total"]
)

print(
    f"Valores distintos com distância 1: "
    f"{qtd_valores_distancia_1}"
)

print(
    f"Acidentes potencialmente envolvidos: "
    f"{qtd_acidentes_distancia_1}"
)

display(candidatos_distancia_1)

Valores distintos com distância 1: 30
Acidentes potencialmente envolvidos: 126


bairro_renaest,qtd_acidentes,bairro_oficial_candidato,codbairro,codra,regiao_adm,distancia
OSWALDO CRUZ,68,Osvaldo Cruz,088,15,MADUREIRA,1
VILA COSMOS,14,Vila Kosmos,072,14,IRAJA,1
RECREIODOS BANDEIRANTES,13,Recreio dos Bandeirantes,132,24,BARRA DA TIJUCA,1
SANTA TEREZA,3,Santa Teresa,014,23,SANTA TEREZA,1
ITANHAGA,2,Itanhangá,127,24,BARRA DA TIJUCA,1
RIO CUMPRIDO,2,Rio Comprido,007,3,RIO COMPRIDO,1
ABLICAO,1,Abolição,070,13,MEIER,1
ANCHETA,1,Anchieta,107,22,ANCHIETA,1
BARA DA TIJUCA,1,Barra da Tijuca,128,24,BARRA DA TIJUCA,1
BARRA DE GUARITIBA,1,Barra de Guaratiba,152,26,GUARATIBA,1


In [0]:
# Construção da dimensão Região Administrativa
# Fonte: IPP / Prefeitura da Cidade do Rio de Janeiro

dim_regiao_validas = (
    df_bairros_ipp
    .select(
        F.col("codra").cast("int").alias("id_regiao"),
        F.col("regiao_adm").alias("regiao_administrativa"),
        F.col("area_plane").cast("int").alias("area_planejamento")
    )
    .filter(F.col("codra").isNotNull())
    .distinct()
)

# Membro especial para acidentes cuja RA não pode ser determinada
dim_regiao_nao_informada = spark.createDataFrame(
    [(-1, "NAO INFORMADO", None)],
    "id_regiao INT, regiao_administrativa STRING, area_planejamento INT"
)

dim_regiao = (
    dim_regiao_nao_informada
    .unionByName(dim_regiao_validas)
)

print("Dimensão Região Administrativa construída com sucesso.")

Dimensão Região Administrativa construída com sucesso.


In [0]:
# Validação da dimensão Região Administrativa

print("Membro especial:")
display(
    dim_regiao
    .filter(F.col("id_regiao") == -1)
)

print("Regiões Administrativas:")
display(
    dim_regiao
    .filter(F.col("id_regiao") != -1)
    .orderBy("id_regiao")
)

# Quantidade de registros
total_regioes = dim_regiao.count()

# Duplicidade da chave
chaves_duplicadas = (
    dim_regiao
    .groupBy("id_regiao")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Mesmo código associado a mais de um nome/Área de Planejamento
inconsistencias = (
    dim_regiao
    .filter(F.col("id_regiao") != -1)
    .groupBy("id_regiao")
    .agg(
        F.countDistinct("regiao_administrativa").alias("qtd_nomes"),
        F.countDistinct("area_planejamento").alias("qtd_aps")
    )
    .filter(
        (F.col("qtd_nomes") > 1) |
        (F.col("qtd_aps") > 1)
    )
    .count()
)

print(f"Quantidade total de registros: {total_regioes}")
print(f"Chaves id_regiao duplicadas: {chaves_duplicadas}")
print(f"Inconsistências código/nome/AP: {inconsistencias}")

Membro especial:


id_regiao,regiao_administrativa,area_planejamento
-1,NAO INFORMADO,null


Regiões Administrativas:


id_regiao,regiao_administrativa,area_planejamento
1,PORTUARIA,1
2,CENTRO,1
2,CENTRO,1
3,RIO COMPRIDO,1
4,BOTAFOGO,2
5,COPACABANA,2
6,LAGOA,2
7,SAO CRISTOVAO,1
8,TIJUCA,2
9,VILA ISABEL,2


Quantidade total de registros: 37
Chaves id_regiao duplicadas: 3
Inconsistências código/nome/AP: 3


In [0]:
# Diagnóstico das chaves de Região Administrativa inconsistentes

diagnostico_regiao = (
    dim_regiao
    .filter(F.col("id_regiao") != -1)
    .groupBy("id_regiao")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.collect_set("regiao_administrativa").alias("nomes"),
        F.collect_set("area_planejamento").alias("areas_planejamento")
    )
    .filter(
        (F.col("qtd_registros") > 1) |
        (F.size("nomes") > 1) |
        (F.size("areas_planejamento") > 1)
    )
    .orderBy("id_regiao")
)

display(diagnostico_regiao)

id_regiao,qtd_registros,nomes,areas_planejamento
2,2,"List(CENTRO , CENTRO)",List(1)
11,2,"List(PENHA , PENHA)",List(3)
17,2,"List(BANGU , BANGU)",List(5)


In [0]:
# Padronização final da dimensão Região Administrativa

dim_regiao_validas_padronizadas = (
    df_bairros_ipp
    .select(
        F.col("codra").cast("int").alias("id_regiao"),

        F.upper(
            F.trim(F.col("regiao_adm"))
        ).alias("regiao_administrativa"),

        F.col("area_plane")
        .cast("int")
        .alias("area_planejamento")
    )
    .filter(F.col("id_regiao").isNotNull())
    .distinct()
)

dim_regiao = (
    dim_regiao_nao_informada
    .unionByName(dim_regiao_validas_padronizadas)
)

print("Dimensão Região Administrativa padronizada.")

Dimensão Região Administrativa padronizada.


In [0]:
# Validação final da dimensão Região Administrativa

total_regioes = dim_regiao.count()

chaves_duplicadas = (
    dim_regiao
    .groupBy("id_regiao")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

inconsistencias = (
    dim_regiao
    .filter(F.col("id_regiao") != -1)
    .groupBy("id_regiao")
    .agg(
        F.countDistinct("regiao_administrativa").alias("qtd_nomes"),
        F.countDistinct("area_planejamento").alias("qtd_aps")
    )
    .filter(
        (F.col("qtd_nomes") > 1) |
        (F.col("qtd_aps") > 1)
    )
    .count()
)

print(f"Quantidade total de registros: {total_regioes}")
print(f"Chaves id_regiao duplicadas: {chaves_duplicadas}")
print(f"Inconsistências código/nome/AP: {inconsistencias}")

display(
    dim_regiao
    .orderBy("id_regiao")
)

Quantidade total de registros: 34
Chaves id_regiao duplicadas: 0
Inconsistências código/nome/AP: 0


id_regiao,regiao_administrativa,area_planejamento
-1,NAO INFORMADO,null
1,PORTUARIA,1
2,CENTRO,1
3,RIO COMPRIDO,1
4,BOTAFOGO,2
5,COPACABANA,2
6,LAGOA,2
7,SAO CRISTOVAO,1
8,TIJUCA,2
9,VILA ISABEL,2


In [0]:
# Persistência da dimensão Região Administrativa na camada Gold

(
    dim_regiao.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_regiao")
)

print("Tabela workspace.gold.dim_regiao gravada com sucesso.")

Tabela workspace.gold.dim_regiao gravada com sucesso.


In [0]:
# Preparação inicial da fato_acidentes
# Granularidade: 1 registro = 1 acidente

fato_base = (
    df_acidentes_silver

    # Chave da dimensão Tempo: AAAAMMDD
    .withColumn(
        "id_tempo",
        F.date_format(
            F.col("data_acidente"),
            "yyyyMMdd"
        ).cast("int")
    )

    # Chave da dimensão Horário: HHMMSS
    .withColumn(
        "id_horario",
        (
            F.hour("hora_acidente") * 10000
            + F.minute("hora_acidente") * 100
            + F.second("hora_acidente")
        ).cast("int")
    )
)

print("Chaves Tempo e Horário preparadas.")

Chaves Tempo e Horário preparadas.


In [0]:
# Validação das chaves Tempo e Horário

total_fato_base = fato_base.count()

# Verificação de chaves nulas
tempo_nulo = (
    fato_base
    .filter(F.col("id_tempo").isNull())
    .count()
)

horario_nulo = (
    fato_base
    .filter(F.col("id_horario").isNull())
    .count()
)

# Verificação de correspondência com dim_tempo
tempo_sem_correspondencia = (
    fato_base.alias("f")
    .join(
        dim_tempo.select("id_tempo").alias("d"),
        F.col("f.id_tempo") == F.col("d.id_tempo"),
        "left_anti"
    )
    .count()
)

# Verificação de correspondência com dim_horario
horario_sem_correspondencia = (
    fato_base.alias("f")
    .join(
        dim_horario.select("id_horario").alias("d"),
        F.col("f.id_horario") == F.col("d.id_horario"),
        "left_anti"
    )
    .count()
)

print(f"Total de acidentes na fato_base: {total_fato_base}")
print(f"id_tempo nulo: {tempo_nulo}")
print(f"id_horario nulo: {horario_nulo}")
print(f"id_tempo sem correspondência na dimensão: {tempo_sem_correspondencia}")
print(f"id_horario sem correspondência na dimensão: {horario_sem_correspondencia}")

Total de acidentes na fato_base: 55046
id_tempo nulo: 0
id_horario nulo: 0
id_tempo sem correspondência na dimensão: 0
id_horario sem correspondência na dimensão: 0


In [0]:
# Associação da dimensão Bairro à fato

fato_com_bairro = (
    fato_base.alias("f")
    .join(
        dim_bairro
        .filter(F.col("id_bairro") != -1)
        .select("id_bairro", "bairro")
        .alias("b"),
        F.col("f.bairro_acidente") == F.col("b.bairro"),
        "left"
    )
    .withColumn(
        "id_bairro_final",
        F.coalesce(
            F.col("b.id_bairro"),
            F.lit(-1)
        )
    )
)

print("Chave Bairro associada à fato.")

Chave Bairro associada à fato.


In [0]:
# Validação da associação da dimensão Bairro

total_com_bairro = fato_com_bairro.count()

bairro_nao_informado = (
    fato_com_bairro
    .filter(F.col("id_bairro_final") == -1)
    .count()
)

bairro_associado = (
    fato_com_bairro
    .filter(F.col("id_bairro_final") != -1)
    .count()
)

# Integridade referencial da chave final
bairro_sem_correspondencia_dim = (
    fato_com_bairro
    .select(
        F.col("id_bairro_final").alias("id_bairro")
    )
    .distinct()
    .join(
        dim_bairro.select("id_bairro"),
        "id_bairro",
        "left_anti"
    )
    .count()
)

print(f"Total após associação com Bairro: {total_com_bairro}")
print(f"Acidentes com bairro associado: {bairro_associado}")
print(f"Acidentes com id_bairro = -1: {bairro_nao_informado}")
print(
    f"Chaves sem correspondência em dim_bairro: "
    f"{bairro_sem_correspondencia_dim}"
)

print(
    f"Percentual com bairro associado: "
    f"{(bairro_associado / total_com_bairro) * 100:.2f}%"
)

Total após associação com Bairro: 55046
Acidentes com bairro associado: 28152
Acidentes com id_bairro = -1: 26894
Chaves sem correspondência em dim_bairro: 0
Percentual com bairro associado: 51.14%


In [0]:
# Associação da Região Administrativa à fato
# Regra: correspondência exata após normalização determinística do bairro

fato_com_regiao = (
    fato_com_bairro.alias("f")
    
    # Normalização usada somente para associação com a referência IPP
    .withColumn(
        "bairro_match_regiao",
        F.upper(
            F.translate(
                F.trim(F.col("f.bairro_acidente")),
                "ÁÀÃÂÉÊÍÓÔÕÚÜÇáàãâéêíóôõúüç",
                "AAAAEEIOOOUUCaaaaeeiooouuc"
            )
        )
    )
    
    .join(
        bairros_ipp_match
        .select(
            "bairro_match",
            F.col("codra").cast("int").alias("id_regiao_ipp")
        )
        .distinct()
        .alias("r"),
        F.col("bairro_match_regiao") == F.col("r.bairro_match"),
        "left"
    )
    
    .withColumn(
        "id_regiao_final",
        F.coalesce(
            F.col("r.id_regiao_ipp"),
            F.lit(-1)
        )
    )
)

print("Chave Região Administrativa associada à fato.")

Chave Região Administrativa associada à fato.


In [0]:
# Validação da associação da dimensão Região Administrativa

total_com_regiao = fato_com_regiao.count()

regiao_associada = (
    fato_com_regiao
    .filter(F.col("id_regiao_final") != -1)
    .count()
)

regiao_nao_informada = (
    fato_com_regiao
    .filter(F.col("id_regiao_final") == -1)
    .count()
)

# Integridade referencial da chave final
regiao_sem_correspondencia_dim = (
    fato_com_regiao
    .select(
        F.col("id_regiao_final").alias("id_regiao")
    )
    .distinct()
    .join(
        dim_regiao.select("id_regiao"),
        "id_regiao",
        "left_anti"
    )
    .count()
)

print(f"Total após associação com Região: {total_com_regiao}")
print(f"Acidentes com RA associada: {regiao_associada}")
print(f"Acidentes com id_regiao = -1: {regiao_nao_informada}")
print(
    f"Chaves sem correspondência em dim_regiao: "
    f"{regiao_sem_correspondencia_dim}"
)
print(
    f"Percentual com RA associada: "
    f"{(regiao_associada / total_com_regiao) * 100:.2f}%"
)

Total após associação com Região: 55046
Acidentes com RA associada: 27075
Acidentes com id_regiao = -1: 27971
Chaves sem correspondência em dim_regiao: 0
Percentual com RA associada: 49.19%


In [0]:
# Construção da tabela fato
# Granularidade: 1 registro = 1 acidente

fato_acidentes = (
    fato_com_regiao
    .select(
        # Identificador do evento
        F.col("f.num_acidente").alias("num_acidente"),

        # Chaves dimensionais
        F.col("f.id_tempo").alias("id_tempo"),
        F.col("f.id_horario").alias("id_horario"),
        F.col("id_bairro_final").cast("long").alias("id_bairro"),
        F.col("id_regiao_final").cast("int").alias("id_regiao"),

        # Medidas
        F.col("f.qtde_acidente").alias("qtde_acidente"),
        F.col("f.qtde_acid_com_obitos").alias("qtde_acid_com_obitos"),
        F.col("f.qtde_envolvidos").alias("qtde_envolvidos"),
        F.col("f.qtde_feridosilesos").alias("qtde_feridosilesos"),
        F.col("f.qtde_obitos").alias("qtde_obitos"),

        # Atributos descritivos do acidente
        F.col("f.tp_acidente").alias("tipo_acidente"),
        F.col("f.cond_meteorologica").alias("condicao_meteorologica"),
        F.col("f.cond_pista").alias("condicao_pista"),
        F.col("f.fase_dia").alias("fase_dia"),
        F.col("f.tp_pista").alias("tipo_pista"),
        F.col("f.tp_pavimento").alias("tipo_pavimento"),
        F.col("f.tp_rodovia").alias("tipo_rodovia")
    )
)

print("Tabela fato_acidentes construída.")

Tabela fato_acidentes construída.


In [0]:
# Validação final da tabela fato_acidentes

total_fato = fato_acidentes.count()

# Unicidade da granularidade
num_acidente_duplicado = (
    fato_acidentes
    .groupBy("num_acidente")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Nulidade das chaves dimensionais
nulos_fks = (
    fato_acidentes
    .select(
        F.sum(F.col("id_tempo").isNull().cast("int")).alias("id_tempo_nulo"),
        F.sum(F.col("id_horario").isNull().cast("int")).alias("id_horario_nulo"),
        F.sum(F.col("id_bairro").isNull().cast("int")).alias("id_bairro_nulo"),
        F.sum(F.col("id_regiao").isNull().cast("int")).alias("id_regiao_nulo")
    )
)

# Integridade referencial das quatro dimensões
tempo_orfao = (
    fato_acidentes.select("id_tempo").distinct()
    .join(dim_tempo.select("id_tempo"), "id_tempo", "left_anti")
    .count()
)

horario_orfao = (
    fato_acidentes.select("id_horario").distinct()
    .join(dim_horario.select("id_horario"), "id_horario", "left_anti")
    .count()
)

bairro_orfao = (
    fato_acidentes.select("id_bairro").distinct()
    .join(dim_bairro.select("id_bairro"), "id_bairro", "left_anti")
    .count()
)

regiao_orfao = (
    fato_acidentes.select("id_regiao").distinct()
    .join(dim_regiao.select("id_regiao"), "id_regiao", "left_anti")
    .count()
)

print(f"Total de registros da fato: {total_fato}")
print(f"num_acidente duplicado: {num_acidente_duplicado}")

print("\nNulidade das chaves:")
display(nulos_fks)

print("Integridade referencial:")
print(f"id_tempo órfão: {tempo_orfao}")
print(f"id_horario órfão: {horario_orfao}")
print(f"id_bairro órfão: {bairro_orfao}")
print(f"id_regiao órfão: {regiao_orfao}")

Total de registros da fato: 55046
num_acidente duplicado: 0

Nulidade das chaves:


id_tempo_nulo,id_horario_nulo,id_bairro_nulo,id_regiao_nulo
0,0,0,0


Integridade referencial:
id_tempo órfão: 0
id_horario órfão: 0
id_bairro órfão: 0
id_regiao órfão: 0


In [0]:
# Persistência da tabela fato na camada Gold

(
    fato_acidentes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.fato_acidentes")
)

print("Tabela workspace.gold.fato_acidentes gravada com sucesso.")

Tabela workspace.gold.fato_acidentes gravada com sucesso.


In [0]:
# Validação física das tabelas da camada Gold

tabelas_gold = [
    "workspace.gold.dim_tempo",
    "workspace.gold.dim_horario",
    "workspace.gold.dim_bairro",
    "workspace.gold.dim_regiao",
    "workspace.gold.fato_acidentes"
]

for tabela in tabelas_gold:
    quantidade = spark.table(tabela).count()
    print(f"{tabela}: {quantidade} registros")

workspace.gold.dim_tempo: 2434 registros
workspace.gold.dim_horario: 1319 registros
workspace.gold.dim_bairro: 311 registros
workspace.gold.dim_regiao: 34 registros
workspace.gold.fato_acidentes: 55046 registros
